## Understanding Cardinality — What Does a Row Represent?

**Cardinality** describes the grain of your dataset: what one row corresponds to in the real world.

Getting this wrong leads to double-counting, broken joins, and inflated metrics. Before writing any query, always ask:

> *"What uniquely identifies a single row in this table?"*

> **Note:** All data in this module is **entirely synthetic** and does not represent any real schools, pupils, or individuals.

Common examples:

| Grain | One row = | Typical identifying columns |
| --- | --- | --- |
| **One row per pupil** | A single pupil | `pupil_id` |
| **One row per pupil per school** | A pupil’s enrolment at a specific school | `pupil_id`, `school_urn` |
| **One row per pupil per school per term** | A pupil’s enrolment in a given term | `pupil_id`, `school_urn`, `term` |

The more columns needed to identify a row, the **finer** the grain. Aggregating to a coarser grain (e.g. from pupil-term to pupil) requires explicit decisions about how to summarise — do you take the latest term, sum across terms, or pick the mode?

### How to check cardinality

The simplest test: count total rows versus distinct combinations of the columns you believe form the grain. If the numbers differ, you have duplicates at that grain.

We’ll use the `pupils_autumn_2024` table from `catalog_40_copper_analyst_training.messy_data` to demonstrate this.

In [0]:
-- Preview the pupils_autumn_2024 table
-- What grain do we think this table is? One row per pupil? Per pupil per school?
SELECT * FROM catalog_40_copper_analyst_training.messy_data.pupils_autumn_2024;

In [0]:
-- Basic cardinality check: is it one row per pupil_id?
-- If total_rows > distinct_pupil_ids, there are duplicates at the pupil grain.
SELECT
  COUNT(*) as total_rows
  ,COUNT(DISTINCT pupil_id) as distinct_pupil_ids
  ,COUNT(*) - COUNT(DISTINCT pupil_id) as duplicate_rows
  ,CASE
    WHEN COUNT(*) = COUNT(DISTINCT pupil_id)
    THEN 'Clean: one row per pupil'
    ELSE 'Duplicates exist at pupil grain'
  END as cardinality_check
FROM catalog_40_copper_analyst_training.messy_data.pupils_autumn_2024;

In [0]:
-- Investigate: which pupil_ids appear more than once, and what differs between them?
SELECT
  pupil_id
  ,COUNT(*) as occurrences
  ,COUNT(DISTINCT first_name) as distinct_first_names
  ,COUNT(DISTINCT school_urn) as distinct_schools
FROM catalog_40_copper_analyst_training.messy_data.pupils_autumn_2024
GROUP BY pupil_id
HAVING COUNT(*) > 1
ORDER BY occurrences DESC;

In [0]:
-- What about a finer grain? Check pupil_id + school_urn as composite key
SELECT
  COUNT(*) as total_rows
  ,COUNT(DISTINCT concat(pupil_id, '-', school_urn)) as distinct_pupil_school
  ,COUNT(*) - COUNT(DISTINCT concat(pupil_id, '-', school_urn)) as duplicate_rows
  ,CASE
    WHEN COUNT(*) = COUNT(DISTINCT concat(pupil_id, '-', school_urn))
    THEN 'Clean: one row per pupil+school'
    ELSE 'Duplicates exist at pupil+school grain'
  END as cardinality_check
FROM catalog_40_copper_analyst_training.messy_data.pupils_autumn_2024;

### Going further: checking cardinality dynamically

The manual approach requires you to know (or guess) which columns form the key. When exploring an unfamiliar table, you can use `INFORMATION_SCHEMA.COLUMNS` to **list all columns** and then systematically test candidate keys.

The query below retrieves the full column list for `pupils_autumn_2024`, which you can then use to construct cardinality checks without hard-coding column names.

In [0]:
-- List all columns in the pupils table from INFORMATION_SCHEMA
-- This lets you discover candidate key columns without inspecting the data manually
SELECT
  column_name
  ,data_type
  ,ordinal_position
FROM catalog_40_copper_analyst_training.information_schema.columns
WHERE table_schema = 'messy_data'
  AND table_name = 'pupils_autumn_2024'
ORDER BY ordinal_position;

In [0]:
-- Dynamic cardinality check: test every individual column as a candidate single-column key
-- Columns where distinct_count = total_rows are unique (potential single-column keys)

DECLARE cardinality_sql STRING;

SET VAR cardinality_sql = (
  SELECT concat(
    'SELECT * FROM (VALUES '
    ,aggregate(
      collect_list(
        concat(
          '((SELECT COUNT(DISTINCT `', column_name, '`) FROM catalog_40_copper_analyst_training.messy_data.pupils_autumn_2024), '''
          ,column_name, ''''
          ,')'
        )
      )
      ,''
      ,(acc, x) -> CASE WHEN acc = '' THEN x ELSE concat(acc, ', ', x) END
    )
    ,') AS t(distinct_count, column_name) ORDER BY distinct_count DESC'
  )
  FROM catalog_40_copper_analyst_training.information_schema.columns
  WHERE table_schema = 'messy_data'
    AND table_name = 'pupils_autumn_2024'
);

EXECUTE IMMEDIATE cardinality_sql;